# This notebook holds a comprehensive spectral analisis of the data. 
It loads the processed data and analise its spectral properties. The main analisys are:
- spectral information: it computes the STFT and spectorgrams using the engine available in layers.TacotronSTFT, and it plots the spectrograms for a few samples as well as perform a PCA on the spectrograms to visualize the main components of the data.
- spectral statistics: it computes the mean and variance of the spectrograms across the dataset, and it plots the distribution of these statistics to understand the variability in the data.
- spectral clustering: it applies a clustering algorithm (e.g., K-means) on the spectrograms to identify potential clusters in the data, and it visualizes the clusters in a 2D space using PCA or t-SNE.
- As a future work, it will load the VERBO dataset and perform the same spectral analysis on it, comparing the results with the current dataset to identify similarities and differences in their spectral properties.

In [ ]:
# ==========================================
# 1. SETUP AND IMPORTS
# ==========================================
import os
import sys
import csv
from pathlib import Path
import random

import torch
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.notebook import tqdm

from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.cluster import KMeans

# Set project root and paths
PROJECT_ROOT = Path.cwd().parent
os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT / "src"))
sys.path.insert(0, str(PROJECT_ROOT / "src" / "models" / "tacotron2_vae"))

# Import TacotronSTFT engine from the repository
from layers import TacotronSTFT

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
print(f"Project root: {PROJECT_ROOT}")

In [ ]:
# ==========================================
# 2. DATA LOADING & STFT ENGINE INITIALIZATION
# ==========================================

# Initialize STFT engine utilizing standard project hyperparameters
stft_engine = TacotronSTFT(
    filter_length=1024,
    hop_length=256,
    win_length=1024,
    n_mel_channels=80,
    sampling_rate=22050,
    mel_fmin=0.0,
    mel_fmax=8000.0
).to(device)

def load_dataset_metadata(csv_path):
    with open(csv_path, "r", encoding="utf-8") as f:
        return list(csv.DictReader(f))

# Using the LibriSpeech-PT or TTS-Portuguese processed dataset as default
metadata_path = Path("data/processed/libriSpeech-en-tacotron-vae/mels_metadata.csv")
if not metadata_path.exists():
    print(f"Warning: {metadata_path} not found. Attempting to fall back to TTS Portuguese...")
    metadata_path = Path("data/processed/tts_portuguese/mels_metadata.csv")

dataset_metadata = load_dataset_metadata(metadata_path)
print(f"Loaded {len(dataset_metadata)} samples from {metadata_path}")

In [ ]:
# ==========================================
# 3. SPECTRAL INFORMATION (COMPUTATION & VISUALIZATION)
# ==========================================
MAX_WAV_VALUE = 32768.0

def extract_mel_spectrogram(waveform_tensor):
    """Passes waveform through the TacotronSTFT engine to yield the Mel-Spectrogram"""
    audio_norm = waveform_tensor.float() / MAX_WAV_VALUE
    audio_norm = audio_norm.unsqueeze(0).to(device)
    with torch.no_grad():
        melspec = stft_engine.mel_spectrogram(audio_norm)
    return melspec.squeeze(0).cpu()

# Visualize a few random samples
num_samples_to_plot = 3
sample_indices = random.sample(range(len(dataset_metadata)), num_samples_to_plot)

fig, axes = plt.subplots(num_samples_to_plot, 2, figsize=(15, 4 * num_samples_to_plot))

for idx, ax_row in zip(sample_indices, axes):
    data = torch.load(dataset_metadata[idx]["mel_path"], map_location="cpu")
    waveform = data["waveform"].squeeze()
    
    # Compute Spectrogram using the STFT engine
    computed_mel = extract_mel_spectrogram(waveform)
    
    # Plot Waveform
    ax_row[0].plot(waveform.numpy(), color='c', alpha=0.8)
    ax_row[0].set_title(f"Waveform: {dataset_metadata[idx]['utt_id']}")
    ax_row[0].set_ylabel("Amplitude")
    
    # Plot Mel-Spectrogram
    im = ax_row[1].imshow(computed_mel.numpy(), origin='lower', aspect='auto', cmap='magma')
    ax_row[1].set_title(f"Computed Mel-Spectrogram: {dataset_metadata[idx]['utt_id']}")
    ax_row[1].set_ylabel("Mel Bin")
    fig.colorbar(im, ax=ax_row[1])

plt.tight_layout()
plt.show()

In [ ]:
# ==========================================
# 4. FEATURE EXTRACTION FOR STATISTICS & CLUSTERING
# ==========================================
# To perform PCA, Clustering, and Statistics over variable length audio,
# we extract the mean over the time-axis to create a fixed-size 80-dimensional embedding per file.

MAX_SAMPLES = 2000 # Limit to 2000 to prevent OOM errors during TSNE/PCA
samples_to_process = min(MAX_SAMPLES, len(dataset_metadata))
random.seed(42)
selected_metadata = random.sample(dataset_metadata, samples_to_process)

spectral_embeddings = []

print(f"Extracting temporal-averaged mel embeddings for {samples_to_process} samples...")
for row in tqdm(selected_metadata):
    data = torch.load(row["mel_path"], map_location="cpu")
    # We can use the pre-computed mel if available to save time, or recompute
    mel = data.get("mel", None)
    if mel is None:
        mel = extract_mel_spectrogram(data["waveform"].squeeze())
    
    # Collapse time dimension (Mean pooling)
    time_averaged_mel = mel.mean(dim=1).numpy() # Shape: (80,)
    spectral_embeddings.append(time_averaged_mel)

X = np.stack(spectral_embeddings) # Matrix of shape (N, 80)
print(f"Extracted feature matrix shape: {X.shape}")

In [ ]:
# ==========================================
# 5. SPECTRAL STATISTICS
# ==========================================
# Calculate mean and variance across the dataset for each Mel frequency bin
global_mean = np.mean(X, axis=0)
global_variance = np.var(X, axis=0)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5))

ax1.plot(global_mean, marker='o', color='royalblue')
ax1.fill_between(range(80), global_mean - np.sqrt(global_variance), global_mean + np.sqrt(global_variance), color='royalblue', alpha=0.2)
ax1.set_title("Global Mean of Mel-Spectrogram Bins (with Std Dev)")
ax1.set_xlabel("Mel Bin Index (0 = Low Freq, 79 = High Freq)")
ax1.set_ylabel("Mean Log-Magnitude")
ax1.grid(True, alpha=0.3)

ax2.bar(range(80), global_variance, color='darkorange', alpha=0.8)
ax2.set_title("Global Variance of Mel-Spectrogram Bins")
ax2.set_xlabel("Mel Bin Index")
ax2.set_ylabel("Variance")
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# ==========================================
# 6. SPECTRAL CLUSTERING (K-MEANS, PCA, t-SNE)
# ==========================================
NUM_CLUSTERS = 4

print("Running K-Means Clustering...")
kmeans = KMeans(n_clusters=NUM_CLUSTERS, random_state=42, n_init=10)
cluster_labels = kmeans.fit_predict(X)

print("Running PCA...")
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X)
explained_var = pca.explained_variance_ratio_

print("Running t-SNE (this might take a few seconds)...")
tsne = TSNE(n_components=2, perplexity=30, random_state=42, n_iter=1000)
X_tsne = tsne.fit_transform(X)

# Visualizations
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

scatter1 = ax1.scatter(X_pca[:, 0], X_pca[:, 1], c=cluster_labels, cmap='tab10', alpha=0.6, s=15)
ax1.set_title(f"PCA of Spectral Embeddings\n(Explained Var: PC1={explained_var[0]*100:.1f}%, PC2={explained_var[1]*100:.1f}%)")
ax1.set_xlabel("Principal Component 1")
ax1.set_ylabel("Principal Component 2")
ax1.grid(True, alpha=0.3)
legend1 = ax1.legend(*scatter1.legend_elements(), title="Clusters")
ax1.add_artist(legend1)

scatter2 = ax2.scatter(X_tsne[:, 0], X_tsne[:, 1], c=cluster_labels, cmap='tab10', alpha=0.6, s=15)
ax2.set_title("t-SNE of Spectral Embeddings")
ax2.set_xlabel("t-SNE Dimension 1")
ax2.set_ylabel("t-SNE Dimension 2")
ax2.grid(True, alpha=0.3)
legend2 = ax2.legend(*scatter2.legend_elements(), title="Clusters")
ax2.add_artist(legend2)

plt.tight_layout()
plt.show()

In [ ]:
# ==========================================
# 7. FUTURE WORK: VERBO DATASET WRAPPER
# ==========================================
def analyze_dataset(csv_path, max_samples=2000, num_clusters=4):
    """
    Modular wrapper to execute the entire analysis pipeline on a new dataset (e.g. VERBO).
    Usage:
        analyze_dataset("data/processed/verbo_dataset/mels_metadata.csv")
    """
    print(f"\n{'='*50}")
    print(f"Analyzing Dataset: {csv_path}")
    print(f"{'='*50}\n")
    
    if not Path(csv_path).exists():
        print("File not found. Please provide a valid metadata CSV.")
        return
        
    metadata = load_dataset_metadata(csv_path)
    samples = min(max_samples, len(metadata))
    selected = random.sample(metadata, samples)
    
    embeds = []
    for row in tqdm(selected, desc="Extracting Features"):
        data = torch.load(row["mel_path"], map_location="cpu")
        mel = data.get("mel", extract_mel_spectrogram(data["waveform"].squeeze()))
        embeds.append(mel.mean(dim=1).numpy())
        
    X_matrix = np.stack(embeds)
    
    # Statistics
    g_mean = np.mean(X_matrix, axis=0)
    g_var = np.var(X_matrix, axis=0)
    
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    axes[0, 0].plot(g_mean, color='royalblue')
    axes[0, 0].set_title("Global Mean")
    
    axes[0, 1].bar(range(80), g_var, color='darkorange')
    axes[0, 1].set_title("Global Variance")
    
    # Clustering
    labels = KMeans(n_clusters=num_clusters, random_state=42, n_init=10).fit_predict(X_matrix)
    X_pca_new = PCA(n_components=2, random_state=42).fit_transform(X_matrix)
    X_tsne_new = TSNE(n_components=2, perplexity=30, random_state=42).fit_transform(X_matrix)
    
    axes[1, 0].scatter(X_pca_new[:, 0], X_pca_new[:, 1], c=labels, cmap='tab10', s=15)
    axes[1, 0].set_title("PCA Clustering")
    
    axes[1, 1].scatter(X_tsne_new[:, 0], X_tsne_new[:, 1], c=labels, cmap='tab10', s=15)
    axes[1, 1].set_title("t-SNE Clustering")
    
    plt.tight_layout()
    plt.show()

# Uncomment and run this when VERBO data is ready:
# analyze_dataset("data/processed/VERBO_dataset/mels_metadata.csv")